# Training neural networks to explain games data

In [1]:
import haiku as hk
import copy

import jax
from jax import grad, jit, random
from jax import lax
import jax.numpy as jnp
import numpy as np
import optax

import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
from scipy.stats import pearsonr
import seaborn as sns
import datetime


seed = 3000
np.random.seed(seed)
rng = jax.random.PRNGKey(seed)
rng

2024-08-08 15:29:43.664370: W external/xla/xla/service/gpu/nvptx_compiler.cc:744] The NVIDIA driver's CUDA version is 11.4 which is older than the ptxas CUDA version (11.8.89). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Array([   0, 3000], dtype=uint32)

In [2]:
 # NQRE, MLP, GameMLP, MQRE, level1_QRE, levelk_QRE, levelk_QRE_two_etas(best), fullNN, varyk_QRE
TARGET_MODEL = 'MLP'
LEVELK = 2
    
NORM_DATA = False # whether we want to normalize the game features or game matrices
GAME_MAT = True # whether we want to include game matrix as input for NN
GAME_FEAT = False # whether we want to include game features as input for NN
PERM_DATA = False # whether we want the dataset to be permuatation-augmented (False=permutation-augment, True=origin)

train, val, test = 0.8, 0.1, 0.1 # train-validation-test split

Activations = []

### Load the 2p2kgames dataset

In [3]:
jax.config.update("jax_enable_x64", True)

filename = 'human data/games2p2k_main.pickle'
with open(filename, 'rb') as f:
    df = pickle.load(f)
df

,unique_id,game_id,permute,p_up,row11,row12,row21,row22,col11,col12,...,max_other,pure_motive,payoff_variance,inequality,nonzerosum,asymmetry,iter_rational,fudenberg_pareto_eq,dtradeoff_self,dtradeoff_other
0,1,1,origin,0.302326,21.0,10.0,24.0,17.0,2.0,19.0,...,19,0,21.250,5,30,10.25,2,1,18,15
1,1,1,col_switch,0.302326,10.0,21.0,17.0,24.0,19.0,2.0,...,19,0,21.250,5,30,10.25,2,1,18,15
2,1,1,row_switch,0.697674,24.0,17.0,21.0,10.0,6.0,4.0,...,19,0,21.250,5,30,10.25,2,1,18,15
3,1,1,col_row_switch,0.697674,17.0,24.0,10.0,21.0,4.0,6.0,...,19,0,21.250,5,30,10.25,2,1,18,15
4,2,1,origin,0.268293,2.0,6.0,19.0,4.0,21.0,24.0,...,24,0,30.125,-5,30,10.25,2,1,15,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9659,2415,1208,col_row_switch,0.027027,22.0,6.0,22.0,41.0,13.0,31.0,...,35,0,77.125,6,52,17.00,2,1,35,11
9660,2416,1208,origin,0.468750,20.0,31.0,35.0,13.0,41.0,6.0,...,41,0,75.625,-6,52,17.00,2,1,11,35
9661,2416,1208,col_switch,0.468750,31.0,20.0,13.0,35.0,6.0,41.0,...,41,0,75.625,-6,52,17.00,2,1,11,35
9662,2416,1208,row_switch,0.531250,35.0,13.0,20.0,31.0,22.0,22.0,...,41,0,75.625,-6,52,17.00,2,1,11,35


In [4]:
df.columns

Index(['unique_id', 'game_id', 'permute', 'p_up', 'row11', 'row12', 'row21',
       'row22', 'col11', 'col12', 'col21', 'col22', 'logrt_norm', 'role',
       'row_topology', 'col_topology', 'ds_self', 'ds_other', 'dissim_self',
       'dissim_other', 'num_psne', 'num_msne', 'pareto_eq', 'pareto_noneq',
       'max_self', 'max_other', 'pure_motive', 'payoff_variance', 'inequality',
       'nonzerosum', 'asymmetry', 'iter_rational', 'fudenberg_pareto_eq',
       'dtradeoff_self', 'dtradeoff_other'],
      dtype='object')

In [5]:
def normalize_features(df, cols, add_target=True):
    features = df[cols]
    sigma = features.std()
    if any(sigma == 0):
        print(sigma)
        raise RuntimeError(
            "A poor data stratification has led to no variation in one of the data "
            "splits for at least one feature (ie std=0). Restratify and try again.")
    mu = features.mean()
    normed_df = (features - mu) / sigma
    if add_target:
        normed_df[target] = df[target]
    return normed_df

### Control the number of features fed into the model by inserting/subtracting items from the list


In [6]:
display_name_from_short_name = {
#         'role': "role",
#         'row_topology': "row_topology",
#         'col_topology': "col_topology",
        'ds_self': "ds_self",
        'ds_other': "ds_other",
        'dissim_self': "dissim_self",
        'dissim_other': "dissim_other",
        'num_psne': "num_psne",
        'num_msne': "num_msne",
        'pareto_eq': "pareto_eq",
        'pareto_noneq': "pareto_noneq",
        'max_self': "max_self",
        'max_other': "max_other",
        'pure_motive': "pure_motive",
        'payoff_variance': "payoff_variance",
        'nonzerosum': "nonzerosum",
        'iter_rational': "iter_rational", 
        'inequality': "inequality",
        'asymmetry': "asymmetry",
        'fudenberg_pareto_eq':'fudenberg_pareto_eq',
        'dtradeoff_self':'dtradeoff_self',
        'dtradeoff_other':'dtradeoff_other',
#         'logrt_norm': "logrt_norm", # if RT info is provided
    }

In [7]:
unique_ids = np.unique(df.unique_id.values)
print(unique_ids, len(unique_ids))

[   1    2    3 ... 2414 2415 2416] 2416


### Train/Val/Test split done on the level of unique_id

In [8]:
target = ['p_up']

In [9]:
if GAME_MAT and GAME_FEAT:
    column_names =  ['row11', 'row12', 'row21', 'row22', 'col11', 'col12', 'col21', 'col22'] + list(display_name_from_short_name)
elif GAME_MAT and not GAME_FEAT:
    column_names =  ['row11', 'row12', 'row21', 'row22', 'col11', 'col12', 'col21', 'col22']
elif not GAME_MAT and GAME_FEAT:
    column_names = list(display_name_from_short_name) 
else:
    raise TypeError('Specify the input type for neural networks!')


In [10]:
def get_batch(df, cols, size=None):
    batch_df = df if size is None else df.sample(size)
    X = batch_df[cols].to_numpy()
    y = batch_df[target].to_numpy()
    return X, y

In [11]:
batch_size = 64
learning_rate = 1e-3
num_training_steps = 10_000
validation_interval = 100

## Neural Network Classes

In [12]:
# Define the MLP model
class MLP(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        global Activations
        Activations = []
        
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        Activations.append([x])
        x = hk.Linear(self.output_size)(x) # Output layer
        x = jax.nn.sigmoid(x) # turn model predictions into probabilities
        return x
    
    
# Define the MLP_eta model
class MLP_eta(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
#         x = jax.nn.relu(x) # remove negative eta
        return x


# Define the MLP_eta2 model
class MLP_eta2(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
#         x = jax.nn.relu(x) # remove negative eta
        return x


# Define the MLP_k model
class MLP_k(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
        x = jax.nn.softmax(x)
        return x
    
    
# Define the row player's MLP model
class Row_MLP(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
        x = jax.nn.softmax(x)
        return x

    
# Define the row player's MLP model
class Col_MLP(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
        x = jax.nn.softmax(x)
        return x

    
# Define the neural network that can approximate a best-response function
class Soft_BR(hk.Module):
    def __init__(self, output_size):
        super().__init__()
        self.output_size = output_size

    def __call__(self, x):
        x = hk.Linear(300)(x)  # Hidden layer
        x = jax.nn.sigmoid(x)     
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x)   
        x = hk.Linear(300)(x)
        x = jax.nn.sigmoid(x) 
        x = hk.Linear(self.output_size)(x) # Output layer
        x = jax.nn.sigmoid(x)
        return x

In [13]:
def utility_function(x):
    alpha = hk.get_parameter("risk_aversion", shape=[], dtype=jnp.float64, init=jnp.ones) # degree of risk aversion
    x = 1-jnp.exp(-alpha*x) # utility function U(x)=1-e^{-alpha*x}
#     x = jnp.power(x, alpha) # utility function U(x)=x^alpha
    return x


### MLP
$p(up) = NN(game~matrix)$

In [14]:
# Function to initialize the model
def forward_pass(x):
    mlp = MLP(output_size=1)
    return mlp(x)

### Neural QRE model
$\eta = NN(game~matrix)$

$p(up) = QRE(game~matrix, \eta, k)$

In [15]:
def inner_loop_body(carry, _): # the loop that calculates QRE
    g, eta, p, q = carry
    EU_up = jnp.exp(eta * g[0] * q + eta * g[1] * (1 - q))
    EU_down = jnp.exp(eta * g[2] * q + eta * g[3] * (1 - q))
    EU_left = jnp.exp(eta * g[4] * p + eta * g[6] * (1 - p))
    EU_right = jnp.exp(eta * g[5] * p + eta * g[7] * (1 - p))
    p, q = jnp.clip(EU_up/(EU_up+EU_down), 1e-4, 1-1e-4), jnp.clip(EU_left/(EU_left+EU_right), 1e-4, 1-1e-4)
    return (g, eta, p, q), None


def outer_loop_fun(xi): # the loop that calculate each QRE for each item in a batch
    g = xi[:8]
    p, q = 0.5, 0.5
    eta = xi[-1] # last element is the NN-predicted eta
    (g, eta, p, q), _ = lax.scan(inner_loop_body, (g, eta, p, q), None, length=8) #length=number of iter for QRE
    return p

    
def lossy_inner_loop(carry, _):
    g, eta, wei, p, q = carry
    EU_up = jnp.exp(eta * g[0] * q + eta * g[1] * (1 - q))
    EU_down = jnp.exp(eta * g[2] * q + eta * g[3] * (1 - q))
    EU_left = jnp.exp(eta*wei * g[4] * p + eta*wei * g[6] * (1 - p))
    EU_right = jnp.exp(eta*wei * g[5] * p + eta*wei * g[7] * (1 - p))
    p, q = jnp.clip(EU_up/(EU_up+EU_down), 1e-4, 1-1e-4), jnp.clip(EU_left/(EU_left+EU_right), 1e-4, 1-1e-4)
    return (g, eta, wei, p, q), None


def outer_loop_fun_levelk(xi): # the loop that calculate each QRE for each item in a batch
    g = xi[:8]
    p, q = 0.5, 0.5
    wei = xi[-2]
    eta = xi[-1] # last element is the NN-predicted eta
    (g, eta, wei, p, q), _ = lax.scan(lossy_inner_loop, (g, eta, wei, p, q), None, length=LEVELK) #length=number of iter for QRE
    return p


# Neural QRE (equilibrium model)
def NQRE_forward_pass(x):
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) 
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # baseline eta
    eta = eta0 + net_output
    x = utility_function(x)
    X = jnp.hstack((x, eta)) # make sure the first 8 items are game matrix, and the last item is the eta
    pred = lax.map(outer_loop_fun, X)
    return pred.reshape(-1, 1)


def levelkQRE_forward_pass(x):
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta = eta0 + net_output
    x = utility_function(x)
#     wei = jnp.asarray([1.])
    wei = hk.get_parameter("lossy_eta_scale", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    X = jnp.hstack((x, jnp.repeat(wei, x.shape[0]).reshape(x.shape[0], 1), eta)) # make sure the first 8 items are game matrix, and the last item is the eta
    pred = lax.map(outer_loop_fun_levelk, X) # check this level1
    return pred.reshape(-1, 1)


def level1QRE_forward_pass(x):
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) # x[:,8:] first 8 items are always the game matrix
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # baseline eta
    eta = jnp.squeeze(eta0 + net_output)
    x = utility_function(x)
    row11, row12, row21, row22 = x[:,0], x[:,1], x[:,2], x[:,3]
    row_diff = eta*(row11+row12-(row21+row22))*0.5
    pred = jax.nn.sigmoid(row_diff)
    pred = jnp.clip(pred, 1e-4, 1-1e-4)
    return pred.reshape((x.shape[0],1))

### 2NN model
$\eta_{row} = NN_1(game~matrix)$

$\eta_{col} = NN_2(game~matrix)$

$p(up) = QR(game~matrix, \eta_{row}, \eta_{col}, k)$

In [16]:
def lossy_inner_loop_feat(carry, _):
    g, eta_row, eta_col, p, q = carry
    EU_up = jnp.exp(eta_row * g[0] * q + eta_row * g[1] * (1 - q))
    EU_down = jnp.exp(eta_row * g[2] * q + eta_row * g[3] * (1 - q))
    EU_left = jnp.exp(eta_col * g[4] * p + eta_col * g[6] * (1 - p))
    EU_right = jnp.exp(eta_col * g[5] * p + eta_col * g[7] * (1 - p))
    p, q = jnp.clip(EU_up/(EU_up+EU_down), 1e-4, 1-1e-4), jnp.clip(EU_left/(EU_left+EU_right), 1e-4, 1-1e-4)
    return (g, eta_row, eta_col, p, q), None


def outer_loop_fun_levelk_feat(xi): # the loop that calculate each QRE for each item in a batch
    g = xi[:8]
    p, q = 0.5, 0.5
    eta_col = xi[-2]
    eta_row = xi[-1] # last element is the NN-predicted eta
    (g, eta_row, eta_col, p, q), _ = lax.scan(lossy_inner_loop_feat, \
                                              (g, eta_row, eta_col, p, q), None, length=LEVELK) #length=number of iter for QRE
    return p


def levelkQRE_feat_forward_pass(x):
    global Activations
    Activations = []
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta_row = eta0 + net_output
    
    
    mlp2 = MLP_eta2(output_size=1)
    net_output2 = mlp2(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta1 = hk.get_parameter("eta1", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta_col = eta1 + net_output2
    Activations.append([eta_row, eta_col])
    
    x = utility_function(x)
    X = jnp.hstack((x, eta_col, eta_row)) # make sure the first 8 items are game matrix, and the last item is the eta
    pred = lax.map(outer_loop_fun_levelk_feat, X) # check this level1
    return pred.reshape(-1, 1)


### 2NN model
$p(k) = NN_1(game~matrix)$

$p(up) = \sum_{k=1}^4 p(k)\times QR(game~matrix, \eta_{row}, \eta_{col}, k)$

In [17]:
def varyk_inner_loop(carry, _):
    g, eta_row, eta_col, p, q = carry
    EU_up = jnp.exp(eta_row * g[0] * q + eta_row * g[1] * (1 - q))
    EU_down = jnp.exp(eta_row * g[2] * q + eta_row * g[3] * (1 - q))
    EU_left = jnp.exp(eta_col * g[4] * p + eta_col * g[6] * (1 - p))
    EU_right = jnp.exp(eta_col * g[5] * p + eta_col * g[7] * (1 - p))
    p, q = jnp.clip(EU_up/(EU_up+EU_down), 1e-4, 1-1e-4), jnp.clip(EU_left/(EU_left+EU_right), 1e-4, 1-1e-4)
    return (g, eta_row, eta_col, p, q), None


def varyk_outer_loop(xi): # the loop that calculate each QRE for each item in a batch
    g = xi[:8]
    p, q = 0.5, 0.5
    wei_k = xi[-3-4:-3]
    eta_col = xi[-2]
    eta_row = xi[-1] # last element is the NN-predicted eta for row
    
    wei_p = wei_k[0]*0.5
    for k in range(3):
        (g, eta_row, eta_col, p, q), _ = lax.scan(varyk_inner_loop, \
                                                  (g, eta_row, eta_col, p, q), None, length=k+1) #length=number of iter for QRE
        wei_p += wei_k[k+1]*p
    return wei_p


def varyk_QRE_forward_pass(x):
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter 
    eta_row = jnp.full((x.shape[0],1), eta0)
    eta1 = hk.get_parameter("eta1", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta_col = jnp.full((x.shape[0],1), eta1)
    
    mlp = MLP_k(output_size=4) # for 4 possible k values (k=0,1,2,3)
    net_output = mlp(x) # x[:,8:] means ignoring the game matrix as input for eta
    k = net_output
    
    x = utility_function(x)
    X = jnp.hstack((x, k, eta_col, eta_row)) # make sure the first 8 items are game matrix, and the last item is the eta
    pred = lax.map(varyk_outer_loop, X) 
    return pred.reshape(-1, 1)


### 3NN model
$\eta_{row} = NN_1(game~matrix)$

$\eta_{col} = NN_2(game~matrix)$

$p(k)=NN_3(game~matrix)$

$p(up)=\sum_k p(k) QR(\eta_{row}, \eta_{col}, k)$

In [18]:
def fullnn_lossy_inner_loop_feat(carry, _):
    g, eta_row, eta_col, p, q = carry
    EU_up = jnp.exp(eta_row * g[0] * q + eta_row * g[1] * (1 - q))
    EU_down = jnp.exp(eta_row * g[2] * q + eta_row * g[3] * (1 - q))
    EU_left = jnp.exp(eta_col * g[4] * p + eta_col * g[6] * (1 - p))
    EU_right = jnp.exp(eta_col * g[5] * p + eta_col * g[7] * (1 - p))
    p, q = jnp.clip(EU_up/(EU_up+EU_down), 1e-4, 1-1e-4), jnp.clip(EU_left/(EU_left+EU_right), 1e-4, 1-1e-4)
    return (g, eta_row, eta_col, p, q), None


def fullnn_outer_loop(xi): # the loop that calculate each QRE for each item in a batch
    g = xi[:8]
    p, q = 0.5, 0.5
    wei_k = xi[-3-4:-3]
    eta_col = xi[-2]
    eta_row = xi[-1] # last element is the NN-predicted eta for row
    
    wei_p = wei_k[0]*0.5
    for k in range(3):
        (g, eta_row, eta_col, p, q), _ = lax.scan(fullnn_lossy_inner_loop_feat, \
                                                  (g, eta_row, eta_col, p, q), None, length=k+1) #length=number of iter for QRE
        wei_p += wei_k[k+1]*p
    return wei_p


def fullnn_levelkQRE_forward_pass(x):
#     global Activations
#     Activations = []
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta_row = eta0+net_output
#     eta_row = jnp.full((x.shape[0],1), eta0)
    
    
    mlp2 = MLP_eta2(output_size=1)
    net_output2 = mlp2(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta1 = hk.get_parameter("eta1", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta_col = eta1+net_output2
#     eta_col = jnp.full((x.shape[0],1), eta1)
    
    mlp3 = MLP_k(output_size=4) # for 4 possible k values (k=0,1,2,3)
    net_output3 = mlp3(x) # x[:,8:] means ignoring the game matrix as input for eta
    k = net_output3
    
#     Activations.append([eta_row, eta_col, k])
    
    x = utility_function(x)
    X = jnp.hstack((x, k, eta_col, eta_row)) # make sure the first 8 items are game matrix, and the last item is the eta
    pred = lax.map(fullnn_outer_loop, X) 
    return pred.reshape(-1, 1)


### Mixture of 2 QRE

In [19]:
# Function to initialize the model
def MQRE_forward_pass(x):
    mlp = MLP_eta(output_size=1)
    net_output = mlp(x) # x[:,8:] means ignoring the game matrix as input for eta
    eta0 = hk.get_parameter("eta0", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    eta = eta0 + net_output
    
    x = utility_function(x)
    row11, row12, row21, row22 = x[:,0], x[:,1], x[:,2], x[:,3]
    row_diff = row11+row12-(row21+row22)
    row_diff = row_diff.reshape((x.shape[0],1))
    row_diff = eta*row_diff*0.5
    l1_qr_pred = jax.nn.sigmoid(row_diff)
    l1_qr_pred = jnp.clip(l1_qr_pred, 1e-4, 1-1e-4)
    
    wei = hk.get_parameter("lossy_eta_scale", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    X = jnp.hstack((x, jnp.repeat(wei, x.shape[0]).reshape(x.shape[0], 1), eta)) # make sure the first 8 items are game matrix, and the last item is the eta
    l2_qr_pred = lax.map(outer_loop_fun_levelk, X) # check this level1
    l2_qr_pred = l2_qr_pred.reshape((x.shape[0],1))
    
    mixwei = hk.get_parameter("mixture_weight", shape=[], dtype=jnp.float64, init=jnp.zeros) # Free parameter
    pred = l1_qr_pred*mixwei + l2_qr_pred*(1-mixwei)
    return pred.reshape((x.shape[0],1))


### Game-inspired Neural Network models
$h_{row} = NN_1(game~matrix)$

$h_{col} = NN_2(game~matrix)$

$p(up) = NN_3(h_{row} \oplus h_{col})$

In [20]:
# Function to initialize the model
def soft_response_forward_pass(x):
    row_mlp = Row_MLP(output_size=100) # my mental representation of the game board
    col_mlp = Col_MLP(output_size=50) # opponent's mental representation of the game board
    row_rep = row_mlp(x)
    col_rep = col_mlp(x)
    
    combined_input = jnp.concatenate([row_rep, col_rep], axis=-1)
    soft_br = Soft_BR(output_size=1)
    p_up = soft_br(combined_input)
    return p_up

### Select a model to train

In [21]:
if TARGET_MODEL=='NQRE':
    model_NQRE = hk.without_apply_rng(hk.transform(NQRE_forward_pass))
    model = copy.deepcopy(model_NQRE) 
elif TARGET_MODEL=='levelk_QRE':
    model_levelk_QRE = hk.without_apply_rng(hk.transform(levelkQRE_forward_pass))
    model = copy.deepcopy(model_levelk_QRE)
elif TARGET_MODEL=='MLP' or TARGET_MODEL=='MLP_feat':
    model_MLP = hk.without_apply_rng(hk.transform(forward_pass))
    model = copy.deepcopy(model_MLP)
elif TARGET_MODEL=='GameMLP':
    model_GameMLP = hk.without_apply_rng(hk.transform(soft_response_forward_pass))
    model = copy.deepcopy(model_GameMLP)
elif TARGET_MODEL=='MQRE':
    model_MQRE = hk.without_apply_rng(hk.transform(MQRE_forward_pass))
    model = copy.deepcopy(model_MQRE)
elif TARGET_MODEL=='level1_QRE':
    model_QRE_l1_variant = hk.without_apply_rng(hk.transform(level1QRE_forward_pass))
    model = copy.deepcopy(model_QRE_l1_variant)
elif TARGET_MODEL=='levelk_QRE_two_etas':
    model_QRE_feat = hk.without_apply_rng(hk.transform(levelkQRE_feat_forward_pass))
    model = copy.deepcopy(model_QRE_feat)
elif TARGET_MODEL=='fullNN':
    model_full_NN = hk.without_apply_rng(hk.transform(fullnn_levelkQRE_forward_pass))
    model = copy.deepcopy(model_full_NN)
elif TARGET_MODEL=='varyk_QRE':
    model_varyk = hk.without_apply_rng(hk.transform(varyk_QRE_forward_pass))
    model = copy.deepcopy(model_varyk)
else:
    raise TypeError('No model found!')

In [22]:
def loss_fn(params, inputs, labels):
    model_pred = model.apply(params, inputs)
    return jnp.mean((model_pred-labels)**2) # MSE

@jax.jit
def train_step(params, inputs, labels, opt_state):
    grads = jax.grad(loss_fn)(params, inputs, labels)
    updates, opt_state = optimizer.update(grads, opt_state)
    new_params = optax.apply_updates(params, updates)
    return new_params, opt_state

In [23]:
mean_MSE, mean_R2 = [], []
for niter in range(1):
    #<------------------------------------------------>#
    
    shuffle_uids = np.random.permutation(unique_ids)
    train_ids = shuffle_uids[:int(train*len(unique_ids))]
    val_ids = shuffle_uids[int(train*len(unique_ids)):int((train+val)*len(unique_ids))]
    test_ids = shuffle_uids[int((train+val)*len(unique_ids)):]
    
    # split the games into train/val/test according to their unique_id
    train_df = df[df['unique_id'].isin(train_ids)]
    val_df = df[df['unique_id'].isin(val_ids)]
    test_df = df[df['unique_id'].isin(test_ids)]

    if PERM_DATA:
        train_df = train_df[train_df.permute=='origin']
        val_df = val_df[val_df.permute=='origin']
        test_df = test_df[test_df.permute=='origin']
    
    if NORM_DATA:
        train_df = normalize_features(train_df, column_names)
        val_df = normalize_features(val_df, column_names)
        test_df = normalize_features(test_df, column_names)
    
    #<------------------------------------------------>#
    
    train_X, train_y = get_batch(train_df, column_names, batch_size)
    init_params = model.init(rng, train_X)
    init_model_pred = model.apply(init_params, train_X)

    optimizer = optax.adam(learning_rate) # optimizer=Adam

    # Initialize optimizer state
    opt_state = optimizer.init(init_params)
    params = copy.deepcopy(init_params)

    # Perform a training step
    worse_validation = 0
    best_validation_loss = np.inf
    for i in range(num_training_steps):
        train_X, train_y = get_batch(train_df, column_names, batch_size)
        params, opt_state = train_step(params, train_X, train_y, opt_state)

        if i % validation_interval == 0:
            # Run validation on the full validation dataset.
            validation_X, validation_y = get_batch(val_df, column_names)
            train_loss = loss_fn(params, train_X, train_y)
            validation_loss = loss_fn(params, validation_X, validation_y)
#             print(f"Step count: {i}")
#             print(f"Train loss: {train_loss}")
#             print(f"Validation loss: {validation_loss}")

            if validation_loss > best_validation_loss:
                worse_validation += 1
                if worse_validation>1:
                    print("Validation loss increased. Stopping!")
                    break
            else:
                best_validation_loss = validation_loss

    # Get training results
    trained_params = copy.deepcopy(params)

    test_X, test_y = get_batch(test_df, column_names)
    model_pred_on_test = model.apply(trained_params, test_X)

#     print(model_pred_on_test.shape, test_y.shape)

    r, p_value = pearsonr(model_pred_on_test.flatten(), test_y.flatten())
    mse_error = loss_fn(trained_params, test_X, test_y)
    r_squared = r**2
    print(f"Test set MSE={mse_error}, R2={r_squared}")
    print(f'=========={niter}==========')
    
    mean_MSE.append(mse_error)
    mean_R2.append(r_squared)

print(f'mean MSE:{np.round(np.mean(mean_MSE),4)}, se MSE:{np.round(np.std(mean_MSE)/np.sqrt(10),4)}, \
        mean R2:{np.round(np.mean(mean_R2),4)}, se R2:{np.round(np.std(mean_R2)/np.sqrt(10),4)}')

Validation loss increased. Stopping!
Test set MSE=0.008641703606758878, R2=0.9140327344424274
==========0==========
mean MSE:0.0086, se MSE:0.0,         mean R2:0.914, se R2:0.0
